The spatial correction factor κ is derived from weekday averages; please comment on whether this remains valid for weekends/holidays. If possible, validate κ-corrected class shares at the 64 independent stations, not only total volumes.

In [ ]:
import sys
import os
os.environ['USE_PYGEOS'] = '0'

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from math import sqrt
from scipy import stats
from sklearn.metrics import r2_score 

import seaborn as sns


# import custom modules
sys.path.append('../../utils')
import data_paths
import traffic_counts
import excel_calendar

## Import Data

In [ ]:
# import visum traffic model and clip to ROI

visum = gpd.read_file(data_paths.VISUM_FOLDER_PATH + 'visum_links.gpkg')
roi_polygon = data_paths.MUNICH_BOARDERS_FILE # defines ROI for clipping
roi = gpd.read_file(roi_polygon).to_crs(visum.crs)
visum = gpd.clip(visum, roi)
visum = visum.explode(ignore_index=True)
visum['road_link_id'] = visum['road_link_id'].astype(int)

In [ ]:
# import counting data and subselect valid counts in 2019
cnt_data_filename = data_paths.COMBINED_COUNTING_DATA
cnt_data = pd.read_parquet(cnt_data_filename)
cnt_data = cnt_data[cnt_data['date'].between('2019-01-01', '2019-12-31')]
cnt = cnt_data[cnt_data['valid']]

In [ ]:
# subselect all road links where counting data is available
road_links_with_counter = cnt['road_link_id'].unique()
visum_cnt = visum[visum['road_link_id'].isin(road_links_with_counter)]

In [ ]:
# initialize traffic counts class without timeprofile
cycles = traffic_counts.TrafficCounts(init_timeprofile=False)

## Processing

In [ ]:
# get daily scaling factors and vehicle shares for 2019

dtv = pd.DataFrame()
shares = pd.DataFrame()

for idx in pd.date_range('2019-01-01', '2019-12-31', freq='D'):
    # scaling factors
    _df = pd.DataFrame(cycles.get_daily_scaling_factors(date=idx)).transpose()
    _df['date'] = idx
    dtv = pd.concat([dtv, _df], ignore_index=True)

    # vehicle shares
    _shares = cycles.get_vehicle_share(date = idx)
    _pc = pd.DataFrame(_shares).transpose()
    _pc['date'] = idx
    shares = pd.concat([shares, _pc])

dtv.set_index('date', inplace=True)
shares = shares.reset_index().set_index(['vehicle_class', 'date'])

In [ ]:
# model traffic counts for all road links with counting data

modelled_count = pd.DataFrame()

for idx, row in visum_cnt.iterrows():
    
    rd_id = row['road_link_id']
    hgv_corr = row['hgv_corr']
    lcv_corr = row['lcv_corr']
    
    model_dtv = dtv[row['scaling_road_type']] * row['dtv_SUM'] 
    
    hgv = model_dtv * shares.loc['HGV'][row['scaling_road_type']] * row['hgv_corr']
    lcv = model_dtv * shares.loc['LCV'][row['scaling_road_type']] * row['lcv_corr']
    
    k = (1- (hgv_corr * shares.loc['HGV'][row['scaling_road_type']])-\
        (lcv_corr * shares.loc['LCV'][row['scaling_road_type']])) / \
        (1 - shares.loc['HGV'][row['scaling_road_type']] - shares.loc['LCV'][row['scaling_road_type']])
        
    #k = 1- (hgv_corr * shares.loc['HGV'][row['scaling_road_type']])/ \
    #    (1 - shares.loc['HGV'][row['scaling_road_type']])

    #lcv = model_dtv * shares.loc['LCV'][row['scaling_road_type']] * k
    pc = model_dtv * shares.loc['PC'][row['scaling_road_type']] * k
    mot = model_dtv * shares.loc['MOT'][row['scaling_road_type']] * k
    bus = model_dtv * shares.loc['BUS'][row['scaling_road_type']] * k
    
    df = pd.DataFrame({
        'SUM': model_dtv,
        'HGV': hgv,
        'LCV': lcv,
        'PC': pc,
        'MOT': mot,
        'BUS': bus
    })
    df.index.name = 'date'
    df['road_link_id'] = row['road_link_id']
    
    modelled_count = pd.concat([modelled_count, df.reset_index()], ignore_index=True)

# sum if multiple road links with same id exist (e.g. bidirectional counting station)
modelled_count = modelled_count.groupby(['road_link_id', 'date']).sum().reset_index()
modelled_count = modelled_count.set_index(['road_link_id', 'date'])
modelled_count

In [ ]:
# prepare counted data
counted = cnt.pivot(index=['road_link_id', 'date'], columns='vehicle_class', values='daily_value')
counted

In [ ]:
# combine modelled and counted data
combined = modelled_count.merge(counted, left_index=True, right_index=True, suffixes=('_modelled', '_counted')).reset_index()
combined

In [ ]:
# add day-type and road type information to the dataframe
cal = excel_calendar.Calendar()

combined['day_type'] = combined['date'].apply(lambda x: cal.get_day_type(x))
combined['road_type'] = combined['road_link_id'].map(visum.groupby('road_link_id')['road_type'].first())



In [ ]:
modelled_count.reset_index().nunique()

# Plotting

In [ ]:
colors = dict(zip(combined['road_type'].unique(), sns.color_palette('colorblind')))

In [ ]:
fig, ax = plt.subplots(2,3 ,figsize=(10,6), tight_layout=True)

i = 0
k = 0

for vc in ['SUM', 'PC', 'HGV', 'LCV', 'MOT', 'BUS']:
    
    df = combined.dropna(subset = [f'{vc}_modelled', f'{vc}_counted'])
    df = df[(df[vc+'_modelled'] > 0) & (df[vc+'_counted'] > 0)]
    
    sns.scatterplot(data= df,
                    y=f'{vc}_modelled', x=f'{vc}_counted', s=5, ax=ax[i,k],
                    palette= colors, hue = 'road_type', alpha = 0.8)
    
    ax[i,k].legend_.remove()
    
    ax[i,k].set_title(vc)
    ax[i,k].set_ylabel('Modelled DTV [veh/day]')
    ax[i,k].set_xlabel('Counted DTV [veh/day]')
    
    max_val = max(df[f'{vc}_modelled'].max(), df[f'{vc}_counted'].max())
    ax[i,k].plot([0, max_val], [0, max_val], color='black', linestyle='--')
    ax[i,k].set_xlim(-50, max_val*1.1)
    ax[i,k].set_ylim(-50, max_val*1.1)

    # add stats to plot
    r2 = r2_score(df[f'{vc}_counted'], df[f'{vc}_modelled'])
    n = len(df)
    
    
    ax[i,k].text(0.05, 0.95, f'$R^2$: {r2:.2f} \nn: {n}', transform=ax[i,k].transAxes,
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # change to next subplot
    k += 1
    if k > 2:
        k = 0
        i += 1

# combined legend for all scatterplots
handles, labels = ax[0,0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.07), markerscale = 6, ncols = 4)

## same plot but only for weekend days

In [ ]:
fig, ax = plt.subplots(2,3 ,figsize=(10,6), tight_layout=True)

i = 0
k = 0

for vc in ['SUM', 'PC', 'HGV', 'LCV', 'MOT', 'BUS']:
    
    df = combined.dropna(subset = [f'{vc}_modelled', f'{vc}_counted'])
    df = df[(df[vc+'_modelled'] > 0) & (df[vc+'_counted'] > 0) & (df['day_type'].isin([3,4]))]

    sns.scatterplot(data= df,
                    y=f'{vc}_modelled', x=f'{vc}_counted', s=5, ax=ax[i,k],
                    palette= colors, hue = 'road_type', alpha = 0.8)

    ax[i,k].legend_.remove()
    
    ax[i,k].set_title(vc)
    ax[i,k].set_ylabel('Modelled DTV [veh/day]')
    ax[i,k].set_xlabel('Counted DTV [veh/day]')
    
    max_val = max(df[f'{vc}_modelled'].max(), df[f'{vc}_counted'].max())
    ax[i,k].plot([0, max_val], [0, max_val], color='black', linestyle='--')
    ax[i,k].set_xlim(-50, max_val*1.1)
    ax[i,k].set_ylim(-50, max_val*1.1)
    
    # add stats to plot
    r2 = r2_score(df[f'{vc}_counted'], df[f'{vc}_modelled'])
    n = len(df)
    
    ax[i,k].text(0.05, 0.95, f'$R^2$: {r2:.2f} \nn: {n}', transform=ax[i,k].transAxes,
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # change to next subplot
    k += 1
    if k > 2:
        k = 0
        i += 1

# combined legend for all scatterplots
handles, labels = ax[0,0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.07), markerscale = 4, ncols = 4)


# suptitle
#fig.suptitle('$\kappa$-corrected modelled vs. counted daily traffic volumes on Saturday and Sunday', fontsize=14)
plt.show()

In [ ]:
fig, ax = plt.subplots(2,3 ,figsize=(10,6), tight_layout=True)

i = 0
k = 0

for vc in ['SUM', 'PC', 'HGV', 'LCV', 'MOT', 'BUS']:
    
    df = combined.dropna(subset = [f'{vc}_modelled', f'{vc}_counted'])
    df = df[(df[vc+'_modelled'] > 0) & (df[vc+'_counted'] > 0) & (df['day_type'].isin([0,1,2]))]

    sns.scatterplot(data= df,
                    y=f'{vc}_modelled', x=f'{vc}_counted', s=5, ax=ax[i,k],
                    palette= colors, hue = 'road_type', alpha = 0.8)
    
    ax[i,k].legend_.remove()
    
    ax[i,k].set_title(vc)
    ax[i,k].set_ylabel('Modelled DTV [veh/day]')
    ax[i,k].set_xlabel('Counted DTV [veh/day]')

    max_val = max(df[f'{vc}_modelled'].max(), df[f'{vc}_counted'].max())
    ax[i,k].plot([0, max_val], [0, max_val], color='black', linestyle='--')
    ax[i,k].set_xlim(-50, max_val*1.1)
    ax[i,k].set_ylim(-50, max_val*1.1)
    
    # add stats to plot
    r2 = r2_score(df[f'{vc}_counted'], df[f'{vc}_modelled'])
    n = len(df)
    
    ax[i,k].text(0.05, 0.95, f'$R^2$: {r2:.2f} \nn: {n}', transform=ax[i,k].transAxes,
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # change to next subplot
    k += 1
    if k > 2:
        k = 0
        i += 1

# combined legend for all scatterplots
handles, labels = ax[0,0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.07), markerscale = 4, ncols = 4)


# suptitle
#fig.suptitle('$\kappa$-corrected modelled vs. counted daily traffic volumes on Weekdays', fontsize=14)
plt.show()


In [ ]:
cnt_data[cnt_data['valid']].nunique()